In [ ]:
import os
import pandas as pd
from datetime import datetime
import pydicom
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import csv
import re
from venn import venn


from utils.selection import *


# Source of RAW DICOMs
BERN_PATH='/nasdata/Berner_Daten/Other_Side_Bilddaten' 
# Project data path
ANALYSIS_PATH='/home/bule/berner_daten_analyse/data'

file_path = '/home/bule/berner_daten_analyse/data/clinical/Kopie Olea von Relevent Data Other File 20181005.xlsx'

data = pd.read_excel(file_path)

data.tail()

In [ ]:
#remove the last empty rows -> these are just commentings
data = data.iloc[:578] 
data.tail()

- check how many of the mrs after 3 months overlap with the patient selection

In [ ]:
##### replace this seciton afterwards with the selected ids from the browserapp selection
dwi_tags_df=pd.read_csv(os.path.join(ANALYSIS_PATH,'dwi_through_tags.csv'))
t1_df=pd.read_csv(os.path.join(ANALYSIS_PATH,'t1_df.csv'))
intersect=set(t1_df['Patient_ID']).intersection(set(dwi_tags_df['Patient_ID']))
######


rows_with_nan = data[~data['mRS nach 3 Monaten'].isna()]
pid_with_mrs3=set(rows_with_nan['PID'])
len(set({float(x) for x in intersect}).intersection(pid_with_mrs3))


In [ ]:
data.info() # -> open in datawrangler

In [ ]:
droplist=[25,27,28]
data=data.iloc[~data.index.isin(droplist)] 
data

# Klinische Daten Lisa

In [ ]:

file_path = '/home/bule/berner_daten_analyse/data/clinical/lisa_stroke/data_wide_all_nihss.csv'

data_lisa = pd.read_csv(file_path)

for col in data_lisa.columns:
    print(col)

In [ ]:
data_lisa

# Imputation

In [ ]:
import pandas as pd
import numpy as np
from tabpfn import TabPFNClassifier

data_encoded = pd.get_dummies(data, dummy_na=True)

# Impute missing values using TabPFN
for column in data.columns:
    if data[column].isnull().any():
        print(f"Imputing column: {column}")
        
        target = data[column]
        features = data.drop(columns=[column])
        mask = target.notnull()
        
        clf = TabPFNClassifier(device='cpu')  # Use 'cuda' if GPU is available
        clf.fit(features[mask], target[mask])
        
        missing_mask = ~mask
        predictions = clf.predict(features[missing_mask])
        
        data.loc[missing_mask, column] = predictions

print(data)